In [1]:
import os, shutil, gc, torch

for path in [
    "/kaggle/working/offload",
    "/kaggle/working/lora_adapter",
    "/kaggle/working/nemotron_lora_adapter",
    "/kaggle/working/submission.zip"
]:
    if os.path.exists(path):
        if os.path.isdir(path):
            shutil.rmtree(path)
        else:
            os.remove(path)

gc.collect()
torch.cuda.empty_cache()
print("Cleaned old outputs.")

Cleaned old outputs.


In [2]:
import site

cutlass_pkg_path = "/kaggle/usr/lib/notebooks/ryanholbrook/nvidia-utility-script/nvidia_cutlass_dsl/python_packages/"
site.addsitedir(cutlass_pkg_path)

import kagglehub
import mamba_ssm
import torch
from peft import LoraConfig, get_peft_model, get_peft_model_state_dict, TaskType
from transformers import AutoModelForCausalLM, AutoTokenizer

print("Dependencies working")

/kaggle/usr/lib/notebooks/ryanholbrook/nvidia-utility-script/torch/compiler/__init__.py:148: FutureWarning: torch._dynamo.allow_in_graph is deprecated and will be removed in a future version. Use torch._dynamo.nonstrict_trace instead.
  return torch._dynamo.allow_in_graph(fn)


Dependencies working


In [3]:
import torch
print(torch.cuda.get_device_name(0))
print(torch.cuda.get_device_properties(0).total_memory / 1024**3, "GB")

NVIDIA RTX PRO 6000 Blackwell Server Edition
94.97076416015625 GB


In [4]:
import pandas as pd

TRAIN_PATH = "/kaggle/input/nvidia-nemotron-3-reasoning-challenge/train.csv"
TEST_PATH = "/kaggle/input/nvidia-nemotron-3-reasoning-challenge/test.csv"

train_df = pd.read_csv(TRAIN_PATH)
test_df = pd.read_csv(TEST_PATH)

print(train_df.shape)
print(test_df.shape)
train_df.head()

(9500, 3)
(3, 2)


,id,prompt,answer
0,00066667,"In Alice's Wonderland, a secret bit manipulati...",10010111
1,000b53cf,"In Alice's Wonderland, a secret bit manipulati...",01000011
2,00189f6a,"In Alice's Wonderland, secret encryption rules...",cat imagines book
3,001b24c4,"In Alice's Wonderland, numbers are secretly co...",XXXVIII
4,001c63cb,"In Alice's Wonderland, secret encryption rules...",wizard creates secret


In [5]:
# Configuration
MODEL_PATH = kagglehub.model_download("metric/nemotron-3-nano-30b-a3b-bf16/transformers/default")
OUTPUT_DIR = "/kaggle/working"
LORA_RANK = 32

model = AutoModelForCausalLM.from_pretrained(
    MODEL_PATH,
    device_map="auto",
    trust_remote_code=True,
    dtype=torch.bfloat16
)

print("Model loaded successfully.")

Loading weights:   0%|          | 0/6243 [00:00<?, ?it/s]

Model loaded successfully.


In [6]:
import polars as pl

train = pl.read_csv('/kaggle/input/nvidia-nemotron-3-reasoning-challenge/train.csv')

train.head()

id,prompt,answer
str,str,str
"""00066667""","""In Alice's Wonderland, a secre…","""10010111"""
"""000b53cf""","""In Alice's Wonderland, a secre…","""01000011"""
"""00189f6a""","""In Alice's Wonderland, secret …","""cat imagines book"""
"""001b24c4""","""In Alice's Wonderland, numbers…","""XXXVIII"""
"""001c63cb""","""In Alice's Wonderland, secret …","""wizard creates secret"""


In [7]:
print(f"Initializing LoRA adapter with rank={LORA_RANK}...")

lora_config = LoraConfig(
    r=LORA_RANK,
    lora_alpha=16,
    target_modules=r".*\.(in_proj|out_proj|up_proj|down_proj)$",
    lora_dropout=0.05,
    bias="none",
    task_type=TaskType.CAUSAL_LM,
)

model = get_peft_model(model, lora_config)

model.print_trainable_parameters()

Initializing LoRA adapter with rank=32...


/usr/local/lib/python3.12/dist-packages/torchao/float8/float8_tensor.py:122: FutureWarning: torch._dynamo.allow_in_graph is deprecated and will be removed in a future version. Use torch._dynamo.nonstrict_trace instead.
  @torch._dynamo.allow_in_graph
/usr/local/lib/python3.12/dist-packages/torchao/float8/float8_tensor.py:195: FutureWarning: torch._dynamo.allow_in_graph is deprecated and will be removed in a future version. Use torch._dynamo.nonstrict_trace instead.
  @torch._dynamo.allow_in_graph
/usr/local/lib/python3.12/dist-packages/torchao/float8/float8_scaling_utils.py:90: FutureWarning: torch._dynamo.allow_in_graph is deprecated and will be removed in a future version. Use torch._dynamo.nonstrict_trace instead.
  @torch._dynamo.allow_in_graph
/usr/local/lib/python3.12/dist-packages/torchao/float8/float8_linear.py:60: FutureWarning: torch._dynamo.allow_in_graph is deprecated and will be removed in a future version. Use torch._dynamo.nonstrict_trace instead.
  @torch._dynamo.allow_

trainable params: 880,138,240 || all params: 32,458,075,584 || trainable%: 2.7116


In [8]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH, trust_remote_code=True)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

print("Tokenizer ready")

Tokenizer ready


In [9]:
import importlib

nemotron_module = importlib.import_module("transformers_modules._1.modeling_nemotron_h")
nemotron_module.is_fast_path_available = False

print("Disabled fast Triton path for training stability.")

Disabled fast Triton path for training stability.


In [10]:
import os, shutil, subprocess

SRC = "/kaggle/usr/lib/notebooks/ryanholbrook/nvidia-utility-script/triton/backends/nvidia/bin/ptxas-blackwell"
DST = "/kaggle/working/ptxas-blackwell"

shutil.copy2(SRC, DST)
os.chmod(DST, 0o755)

os.environ["TRITON_PTXAS_BLACKWELL_PATH"] = DST
os.environ["TRITON_PTXAS_PATH"] = DST

print("Copied ptxas-blackwell to:", DST)
print(subprocess.check_output([DST, "--version"]).decode())

Copied ptxas-blackwell to: /kaggle/working/ptxas-blackwell
ptxas-blackwell: NVIDIA (R) Ptx optimizing assembler
Copyright (c) 2005-2025 NVIDIA Corporation
Built on Tue_May_27_02:18:05_PDT_2025
Cuda compilation tools, release 12.9, V12.9.86
Build cuda_12.9.r12.9/compiler.36037853_0



In [11]:
import torch
from torch.utils.data import Dataset, DataLoader

class ReasoningDataset(Dataset):
    def __init__(self, df, tokenizer, max_length=512):
        self.df = df.reset_index(drop=True)
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        prompt = str(self.df.loc[idx, "prompt"])
        answer = str(self.df.loc[idx, "answer"])

        text = f"{prompt}\n\nAnswer: {answer}"

        enc = self.tokenizer(
            text,
            truncation=True,
            max_length=self.max_length,
            padding="max_length",
            return_tensors="pt"
        )

        input_ids = enc["input_ids"].squeeze(0)
        attention_mask = enc["attention_mask"].squeeze(0)

        labels = input_ids.clone()
        labels[attention_mask == 0] = -100

        return {
            "input_ids": input_ids,
            "attention_mask": attention_mask,
            "labels": labels
        }

train_dataset = ReasoningDataset(train_df, tokenizer, max_length=512)
train_loader = DataLoader(train_dataset, batch_size=1, shuffle=True)

model.train()
optimizer = torch.optim.AdamW(model.parameters(), lr=2e-4)

for step, batch in enumerate(train_loader):
    batch = {k: v.to(model.device) for k, v in batch.items()}

    outputs = model(**batch)
    loss = outputs.loss

    loss.backward()
    optimizer.step()
    optimizer.zero_grad()

    print(f"Step {step+1} | Loss: {loss.item():.4f}")

    if step == 4:
        break

Step 1 | Loss: 1.1477
Step 2 | Loss: 2.3032
Step 3 | Loss: 5.8154
Step 4 | Loss: 2.4965
Step 5 | Loss: 2.0955


In [12]:
MAX_STEPS = 2500
GRAD_ACCUM_STEPS = 4

model.train()
optimizer = torch.optim.AdamW(model.parameters(), lr=2e-4)
optimizer.zero_grad()

for step, batch in enumerate(train_loader):
    batch = {k: v.to(model.device) for k, v in batch.items()}

    outputs = model(**batch)
    loss = outputs.loss / GRAD_ACCUM_STEPS
    loss.backward()

    if (step + 1) % GRAD_ACCUM_STEPS == 0:
        optimizer.step()
        optimizer.zero_grad()

    if (step + 1) % 10 == 0:
        print(f"Step {step+1} | Loss: {loss.item() * GRAD_ACCUM_STEPS:.4f}")

    if step + 1 >= MAX_STEPS:
        break

print("2500-step training complete.")

Step 10 | Loss: 1.5548
Step 20 | Loss: 1.2726
Step 30 | Loss: 1.2678
Step 40 | Loss: 4.1535
Step 50 | Loss: 0.6370
Step 60 | Loss: 3.7624
Step 70 | Loss: 0.5926
Step 80 | Loss: 2.1509
Step 90 | Loss: 0.4479
Step 100 | Loss: 0.7010
Step 110 | Loss: 2.2703
Step 120 | Loss: 0.5432
Step 130 | Loss: 3.4338
Step 140 | Loss: 0.5518
Step 150 | Loss: 0.4146
Step 160 | Loss: 3.6177
Step 170 | Loss: 0.4387
Step 180 | Loss: 0.7882
Step 190 | Loss: 3.4326
Step 200 | Loss: 0.7794
Step 210 | Loss: 0.4444
Step 220 | Loss: 0.6783
Step 230 | Loss: 1.0881
Step 240 | Loss: 0.4248
Step 250 | Loss: 0.4576
Step 260 | Loss: 0.3830
Step 270 | Loss: 0.6826
Step 280 | Loss: 0.6642
Step 290 | Loss: 0.3981
Step 300 | Loss: 2.1636
Step 310 | Loss: 0.7509
Step 320 | Loss: 0.4486
Step 330 | Loss: 0.3272
Step 340 | Loss: 3.1780
Step 350 | Loss: 0.7315
Step 360 | Loss: 0.6790
Step 370 | Loss: 0.5515
Step 380 | Loss: 0.6965
Step 390 | Loss: 1.9338
Step 400 | Loss: 0.6753
Step 410 | Loss: 0.7183
Step 420 | Loss: 0.5757
S

In [13]:
OUTPUT_DIR = "/kaggle/working/lora_adapter"
model.save_pretrained(OUTPUT_DIR)
print(f"LoRA adapter saved to {OUTPUT_DIR}")

LoRA adapter saved to /kaggle/working/lora_adapter


In [14]:
import os

for root, dirs, files in os.walk("/kaggle/working"):
    for file in files:
        print(os.path.join(root, file))

/kaggle/working/ptxas-blackwell
/kaggle/working/__notebook__.ipynb
/kaggle/working/lora_adapter/README.md
/kaggle/working/lora_adapter/adapter_model.safetensors
/kaggle/working/lora_adapter/adapter_config.json


In [15]:
import kagglehub

MODEL_PATH = kagglehub.model_download(
    "metric/nemotron-3-nano-30b-a3b-bf16/transformers/default"
)

print(MODEL_PATH)

/kaggle/input/models/metric/nemotron-3-nano-30b-a3b-bf16/transformers/default/1


In [16]:
import os, zipfile

ADAPTER_DIR = "/kaggle/working/lora_adapter"
ZIP_PATH = "/kaggle/working/submission.zip"

with zipfile.ZipFile(ZIP_PATH, "w", zipfile.ZIP_DEFLATED) as zipf:
    for root, dirs, files in os.walk(ADAPTER_DIR):
        for file in files:
            full_path = os.path.join(root, file)
            arcname = os.path.relpath(full_path, ADAPTER_DIR)
            zipf.write(full_path, arcname)

print("Created:", ZIP_PATH)

Created: /kaggle/working/submission.zip


In [17]:
!ls -lh /kaggle/working/submission.zip

-rw-r--r-- 1 root root 3.0G Jun  8 19:05 /kaggle/working/submission.zip


In [18]:
!unzip -l /kaggle/working/submission.zip | head -20

Archive:  /kaggle/working/submission.zip
  Length      Date    Time    Name
---------  ---------- -----   ----
     5312  2026-06-08 19:03   README.md
3522350336  2026-06-08 19:03   adapter_model.safetensors
     1045  2026-06-08 19:03   adapter_config.json
---------                     -------
3522356693                     3 files
